In [0]:
def configure_adls_oauth(storage_account, client_id, client_secret, tenant_id):
    """Configure OAuth (Service Principal) authentication for ADLS Gen2."""
    base = f"{storage_account}.dfs.core.windows.net"
    endpoint = f"https://login.microsoftonline.com/{tenant_id}/oauth2/token"

    spark.conf.set(f"fs.azure.account.auth.type.{base}", "OAuth")
    spark.conf.set(f"fs.azure.account.oauth.provider.type.{base}",
                   "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
    spark.conf.set(f"fs.azure.account.oauth2.client.id.{base}", client_id)
    spark.conf.set(f"fs.azure.account.oauth2.client.secret.{base}", client_secret)
    spark.conf.set(f"fs.azure.account.oauth2.client.endpoint.{base}", endpoint)
    print(f"✔ OAuth configured for: {storage_account}")


def get_adls_path(container, storage_account, path=""):
    """Build an ABFSS URI for ADLS Gen2."""
    base = f"abfss://{container}@{storage_account}.dfs.core.windows.net"
    return f"{base}/{path}" if path else f"{base}/"


def list_adls_files(container, storage_account, path=""):
    """List files/folders at an ADLS Gen2 path."""
    full_path = get_adls_path(container, storage_account, path)
    return dbutils.fs.ls(full_path)


def read_adls_file(container, storage_account, file_path, file_format="csv", **options):
    """Read a file from ADLS Gen2 into a DataFrame."""
    full_path = get_adls_path(container, storage_account, file_path)
    reader = spark.read.format(file_format)
    for key, value in options.items():
        reader = reader.option(key, value)
    return reader.load(full_path)


def write_adls_file(df, container, storage_account, output_path, file_format="parquet", mode="overwrite"):
    """Write a DataFrame to ADLS Gen2."""
    full_path = get_adls_path(container, storage_account, output_path)
    df.write.format(file_format).mode(mode).save(full_path)
    print(f"✔ Data written to: {full_path}")

In [0]:
# --- Configuration (using Databricks Secret Scope for credentials) ---
STORAGE_ACCOUNT = "bronzeadlsstorage"
CONTAINER       = "ainput"
CLIENT_ID       = "fe8fdf73-1c07-4728-8f16-e6bb1fb51951"
CLIENT_SECRET   = dbutils.secrets.get(scope="adls-scope", key="client-secret")
TENANT_ID       = "1c00043c-77d0-4125-ab5a-a486bcc0d7bf"

configure_adls_oauth(STORAGE_ACCOUNT, CLIENT_ID, CLIENT_SECRET, TENANT_ID)

In [0]:
# List root of container
display(list_adls_files(CONTAINER, STORAGE_ACCOUNT))

# List files in ainput1 subfolder
display(list_adls_files(CONTAINER, STORAGE_ACCOUNT, "ainput1"))

In [0]:
df_csv = read_adls_file(CONTAINER, STORAGE_ACCOUNT, "ainput1/Employee2.csv", "csv", header="true")
display(df_csv)

In [0]:
df_json = read_adls_file(CONTAINER, STORAGE_ACCOUNT, "ainput1/employees_single.json", "json")
display(df_json)

In [0]:
df_parquet = read_adls_file(CONTAINER, STORAGE_ACCOUNT, "ainput1/yellow_tripdata_2023-01.parquet", "parquet")
display(df_parquet)

In [0]:
write_adls_file(df_parquet, CONTAINER, STORAGE_ACCOUNT, "output")

In [0]:
# Example: reading with an explicit configs dictionary (for mount-based or other patterns)
# The OAuth is already set via configure_adls_oauth(), so this cell is optional.
df_csv_alt = read_adls_file(CONTAINER, STORAGE_ACCOUNT, "ainput1/Employee2.csv", "csv", header="true")
display(df_csv_alt)

In [0]:
# List all available secret scopes
scopes = dbutils.secrets.listScopes()
display(scopes)

# Uncomment below to list secrets within a specific scope
# display(dbutils.secrets.list("your-scope-name"))

In [0]:
client_secret = dbutils.secrets.get(scope="adls-scope", key="client-secret")
display(client_secret)